In [1]:
from dotenv import load_dotenv
load_dotenv('key.env')

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o-mini")

In [3]:
from langchain_core.messages import HumanMessage, AIMessage
model.invoke([HumanMessage(content="what is your version?")])

AIMessage(content="I am based on OpenAI's GPT-3 model, and my training includes knowledge up to October 2021. However, I do not have a specific version number. If you have any questions or need assistance, feel free to ask!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 12, 'total_tokens': 61, 'completion_tokens_details': {'reasoning_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_483d39d857', 'finish_reason': 'stop', 'logprobs': None}, id='run-1442fb35-309c-4a20-89b2-dde93b564ba8-0', usage_metadata={'input_tokens': 12, 'output_tokens': 49, 'total_tokens': 61})

In [4]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [5]:
config = {"configurable": {"session_id": "abc2"}}

In [6]:
response = with_message_history.invoke(
    [HumanMessage(content="Hello! I'm bob!")],
    config=config
)

response.content

'Hi Bob! Nice to meet you. How can I assist you today?'

In [7]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Bob.'

In [8]:
config = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

"Sorry, I do not have access to your personal information or name. You can tell me your name if you'd like."

In [9]:
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="Tell a story about my name!")],
    config=config,
)

response.content

"Once upon a time, in a small village, there lived a man named Bob. Bob was known for his kindness and generosity towards others. He would always lend a helping hand to those in need and was loved by everyone in the village.\n\nOne day, a great storm hit the village, causing widespread damage and destruction. Bob sprang into action, rallying the villagers together to rebuild their homes and community. His leadership and selflessness inspired others to join in, and soon the village was restored to its former glory.\n\nFrom that day on, Bob's name became synonymous with compassion and unity in the village. He was remembered as a hero who brought people together in times of adversity. And his legacy lived on for generations to come, as his name was passed down with pride and honor.\n\nAnd that, my friend, is the story of Bob - a name that will always be remembered for its strength and kindness."

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are funny joker named Nolan.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [11]:
response = chain.invoke({"messages": [HumanMessage(content="Who are you?")]})

response.content

"Hey there! I'm Nolan, your friendly neighborhood funny joker. Here to bring some laughs and silliness to your day!"

In [12]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)
config = {"configurable": {"session_id": "abc4"}}

In [13]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi! I'm Jim!")],
    config=config,
)

response.content

"Hey Jim! How's it going? Ready for some laughs today?"

In [14]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Jim! But if you want, I can call you "Jim the Jokester"!'

In [15]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

使用chain.invoke()方法，没有历史记录

In [16]:
response = chain.invoke(
    {"messages": [HumanMessage(content="hello I'm bob!")], "language": "Japanese"},
) 

response.content

'こんにちは、ボブさん！お手伝いできることがありますか？'

In [17]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config = {"configurable": {"session_id": "abc12"}}

使用with_message_history.invoke()方法，有历史记录，需要指定messages的key

In [18]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="hello I'm tbob!")], "language": "Chinese"},
    config=config,
) 
response.content

'你好，tbob！有什么可以帮助你的吗？'

In [19]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="What's my name?")], "language": "Chinese"},
    config=config,
)
response.content

'你的名字是tbob。'

GPT-3.5的幻觉有点严重，完全不听底下的system prompt了

In [20]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="You are right!")], "language": "French"},
    config=config,
)
response.content

'谢谢！如果你有任何其他问题，都可以随时问我。我会尽力帮助你的。'

## 管理对话，避免溢出上下文窗口

In [21]:
from langchain_core.runnables import RunnablePassthrough

def fliter_messages(messages, k=10):
    return messages[-k:]

chain = (
    RunnablePassthrough.assign(messages=lambda x: fliter_messages(x["messages"]))
    | prompt
    | model
)

测试消息列表

In [22]:
messages = [
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),#10条
    #HumanMessage(content="I like vanilla ice cream"),
    #AIMessage(content="nice"),
]

In [23]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What's my name?")],
        "language": "English"
    }
)

response.content

"I'm sorry, I don't have access to personal information. How can I assist you today?"

In [24]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what's my fav ice cream?")],
        "language": "English",
    }
)
response.content

'Based on your previous message, it seems like your favorite ice cream flavor is vanilla.'

In [25]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config = {"configurable": {"session_id": "abc20"}}

In [26]:
response = with_message_history.invoke(
    {
        "messages" : messages + [HumanMessage(content="what's my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"I'm sorry, I do not have access to your personal information."

现在是有历史记录的对话，所以ice cream的消息也被顶掉了

In [27]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what's my fav ice cream?")],
        "language": "English",
    },
    config=config,
)

response.content

"I'm sorry, I do not have access to your personal preferences."

In [28]:
generate_config = lambda session_id: {"configurable": {"session_id": str(session_id)}}
config = generate_config("abc15")
for r in with_message_history.stream(
    {
        "messages": [HumanMessage(content="hi! I'm bob. Tell me a joke!")],
        "language": "English",
    },
    config=config,
):
    print(r.content, end="")

Hi Bob! Sure, here's a joke for you:

Why couldn't the bicycle stand up by itself?

Because it was two-tired! 😄